# **TWITTER SENTIMENT ANALYSIS - PROJECT PREDICTION SENTIMENT ANALYSIS**

# Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# WordCloud --> To Show All Words in dataset
from wordcloud import WordCloud as wc

# Regular Expression
import re

# Text Manipulation --> Natural Languages Tool Kit
import nltk

# Tokenizer, word_tokenizer
from nltk.tokenize import word_tokenize

# Stopwords
from nltk.corpus import stopwords

# To Stemming
from nltk.stem import WordNetLemmatizer

# Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# TO LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout, Bidirectional, Conv1D, GlobalMaxPooling1D
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Other
import string
import warnings
import tabulate
import time


pd.set_option('display.max_colwidth', 200)
warnings.filterwarnings('ignore', category=DeprecationWarning)

%matplotlib inline

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


NameError: name 'tf' is not defined

# Gathering Data

In [ ]:
df_train = pd.read_csv('../data/train_E6oV3lV.csv')
df_test = pd.read_csv('../data/test_tweets_anuFYb8.csv')
df_submission = pd.read_csv('../data/sample_submission_gfvA5FD.csv')

display(df_train.head(3))  # With labels
display(df_test.head(3))  # Without labels

print(f'Train Shape : {df_train.shape}, Test Shape : {df_test.shape}')

# Data Preparation

## Inspection Data

In [ ]:
print(f'Labels: {df_train['label'].unique()}')  # [0,1]

print(f'{'='*30} VALUE COUNTS OF THIS LABELS {'='*30}')

print(f'Value Counts : \n{df_train['label'].value_counts()}')

labels_no = df_train[df_train['label'] == 0]
labels_yes = df_train[df_train['label'] == 1]

print(f'{'='*30}  LABELS  {'='*30}')
print(f'{'='*30} Label No {'='*30}')
print(labels_no.head(3).to_markdown())
print(f'{'='*30} Label Yes {'='*30}')
print(labels_yes.head(3).to_markdown())

In [ ]:
len_train = df_train['tweet'].str.len()
len_test = df_test['tweet'].str.len()

plt.hist(len_train, bins=20, label='Train_Tweets')
plt.hist(len_test, bins=20, label='Test_Tweets')
plt.legend()
plt.title('Distribusi Panjang Karakter Tweet')
plt.xlabel('Panjang Karakter')
plt.ylabel('Frekuensi')
plt.show()

Meskipun data_train lebih banyak, tapi ternyata panjang karakter data_train maupun data_test hampir sama.

## Combine Train and Test Datasets

In [ ]:
combine = pd.concat([df_train, df_test], ignore_index = True)
print(combine.shape)

# Preprocessing Data

### Cleaning Text

In [ ]:
# Lower Case
combine['tweet'] = combine['tweet'].str.lower()

# Remove Mention & Link
combine['clean_text'] = combine['tweet'].str.replace(r'http\S+|www\S+|@\w+', '', regex=True)

# Regex 2: Removing Punctuation, Numbers, and Special Character
# Perhatikan kita menambahkan \.,!\?' ke dalam kurung siku
combine['clean_text'] = combine['clean_text'].str.replace(r"[^a-zA-Z#.,!?'\s]", ' ', regex=True)

def normalize_repeated_chars(text):
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

combine['clean_text'] = combine['clean_text'].apply(normalize_repeated_chars)

# Regex 3: Delete double space.
combine['clean_text'] = combine['clean_text'].str.replace(r'\s+', ' ', regex=True)

combine['clean_text'] = combine['clean_text'].str.replace(r'#', '', regex=True)

# 4. Tokenisasi with NLTK
combine['tokens'] = combine['clean_text'].apply(word_tokenize)

## Stemming

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

negation_words = {'no', 'not', 'nor', 'never', 'very'}
stop_words -= negation_words

combine['tokens'] = combine['tokens'].apply(
    lambda tokens: [lemmatizer.lemmatize(word)
                   for word in tokens
                   if word not in stop_words 
                    and len(word) > 2
                    and word != 'amp'
                   ]
)

### Join All tokens back (Detokenized)

In [ ]:
combine['final_text'] = combine['tokens'].apply(lambda x: " ".join(x))

## The Most Common Words In The Entire Datasets --> For Stemmed_Split

In [ ]:
def word_cloud(data):
    return wc(width=800, height=500, random_state=21, max_font_size=110).generate(data)

In [ ]:
non_hatespeech_labels = combine['final_text'][combine['label']== 0]
hatespeech_labels = combine['final_text'][combine['label']== 1]

### All Datasets

In [ ]:
all_words = ' '.join([text for text in combine['final_text']])
wordcloud = word_cloud(all_words)

In [ ]:
plt.figure(figsize=(16,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

### No Hate Speech Datasets

In [ ]:
non_hatespeech_words = ' '.join([text for text in non_hatespeech_labels])
wordcloud_non_hatespeech = word_cloud(positive_words)

In [ ]:
plt.figure(figsize=(16,10))
plt.imshow(wordcloud_non_hatespeech,  interpolation='bilinear')
plt.axis('off')
plt.show()

Jika dilihat, ini sudah sesuai, tidak ada kata-kata hate speech lagi

### Hatespeech Datasets

In [ ]:
hatespeech_words = ' '.join([text for text in hatespeech_labels])
wordcloud_hatespeech = word_cloud(negative_words)

In [ ]:
plt.figure(figsize=(16,10))
plt.imshow(wordcloud_hatespeech, interpolation='bilinear')
plt.axis('off')
plt.show()           

Jika dilihat, kalimat hatespeech ini juga sudah sesuai, seperti obama, trump, black, racist, hate, racism, america, dan lainnya, meskipun ada beberapa data yang masih termasuk kata positif.

# Machine Learning Model

## Prep All Models

In [ ]:
combine.head()

In [ ]:
#  Inisialisasi vectorization
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,3),
    min_df=1,
    max_df=0.9,
    sublinear_tf=True
)

X = tfidf.fit_transform(combine['final_text'])

X = X[:len(df_train)]
y = df_train['label']  # Bebas, karena mau bagaimana pun urutan label tidak akan berubah.

X

In [ ]:
# SPLIT DATA
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Modelling
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    "Linear SVM": LinearSVC(max_iter=2000, random_state=42, C=1.5, class_weight='balanced')
}

print("Mulai melatih model dengan TF-IDF Super...\n" + "="*50)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} | Akurasi: {acc * 100:.2f}%")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

In [ ]:
# 1. AMBIL MODEL TERBAIK KITA (Linear SVM)
best_model = models["Linear SVM"]

# 2. BIKIN KALIMAT UJIAN DADAKAN (Silakan ganti teks bahasa Inggrisnya sesuka hati!)
tes_tweet = [
    "I absolutely love this new phone, the camera is amazing!",  # Harusnya Positif
    "This product is garbage. Worst customer service ever, I hate it!", # Harusnya Negatif
    "Today is a normal day, just drinking coffee", # Harusnya Positif/Netral
    "You are so stupid and ugly, completely useless!", # Harusnya Negatif (Hate Speech)
    "You are a disgusting racist nazi bigot, get out of here!", 
    "This misogyny and discrimination against minority is a treason."
]


# 3. UBAH TEKS JADI ANGKA PAKAI MESIN TF-IDF YANG UDAH JADI
X_tes_dadakan = tfidf.transform(tes_tweet)

# 4. SURUH AI MENEBAK!
prediksi_dadakan = best_model.predict(X_tes_dadakan)

# 5. TAMPILKAN HASILNYA BIAR GAMPANG DIBACA
print("=== HASIL UJIAN DADAKAN AI ===\n")
for teks, hasil in zip(tes_tweet, prediksi_dadakan):
    # Di dataset Twitter Kaggle: 0 = Positif/Netral, 1 = Negatif/Hate Speech
    label_teks = "🟢 POSITIF / NETRAL" if hasil == 0 else "🔴 NEGATIF / HATE SPEECH"
    
    print(f"Tweet: '{teks}'")
    print(f"Tebakan AI: {label_teks}\n")

In [ ]:
feature_names = tfidf.get_feature_names_out()

coefs = model.coef_[0]

top_positive = sorted(zip(coefs, feature_names), reverse=True)[:20]
top_negative = sorted(zip(coefs, feature_names))[:20]

print("Top Positive Words:")
for coef, word in top_positive:
    print(word)

print("\nTop Negative Words:")
for coef, word in top_negative:
    print(word)

Karena datanya terdeteksi hate speech, tetapi ketika disuruh predict, memiliki predict yang jelek, maka kita akan coba beralih ke Deep Learning (USING LSTM)

# LSTM (Deep Learning)

Deep learning mempunyai caranya sendiri, tanpa perlu menggunakan TfId. Mari kita lihat data kita:

In [ ]:
combine['final_text']

In [ ]:
# Samakan jumlah row
X = combine['final_text'][:len(df_train)]
y = df_train['label']

# SET MAX WORD AND MAX LEN
MAX_WORDS = 5000
MAX_LEN = combine['final_text'].apply(lambda x: len(str(x).split())).max()

# TOKENIZER
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')  # OOV artinya kata yang tidak masuk top 5000 akan diganti jadi token 'out of vocabulary'

# Train tokenizer to read and remind all words from data
tokenizer.fit_on_texts(X)

# Converts the entire tweet into a string of index numbers
seq_text = tokenizer.texts_to_sequences(X)

# Padding -> equalize word length
X_pad = pad_sequences(seq_text, maxlen=MAX_LEN, padding='post', truncating='post')

# Split Data
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(X_pad, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# WEIGHT (To imbalanced data)
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_dl), y=y_train_dl)

class_weight = dict(enumerate(weights))

In [ ]:
dl_models = {
    # Petarung 1: LSTM Standar
    "LSTM Standar": Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
        LSTM(64),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ]),

    # Petarung 2: Bidirectional LSTM (Membaca kalimat dari depan & belakang) 🔥
    "Bi-LSTM": Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
        Bidirectional(LSTM(64)), # Tinggal dibungkus pakai Bidirectional()
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ]),

    # Petarung 3: GRU (Versi efisien dari LSTM)
    "GRU Standar": Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
        GRU(64),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ]),

    # Petarung 4: CNN 1D (Jagoan cari pola kata/N-Gram, training super cepat!) ⚡
    "CNN 1D": Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
        Conv1D(filters=128, kernel_size=5, activation='relu'), # Membaca 5 kata sekaligus
        GlobalMaxPooling1D(), # Mengambil pola yang paling kuat dari tiap kalimat
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
}

In [ ]:
history_dict = {}

for name, model in dl_models.items():
    print(f"\n{'='* 20} {name} {'='* 20}")

    early_stop = EarlyStopping(
        monitor='val_loss', 
        patience=2, 
        restore_best_weights=True,
        verbose=1
    )

    custom_adam = Adam(learning_rate=0.0005)
    
    # 1. Compile Model pakai optimizer baru
    model.compile(loss='binary_crossentropy', optimizer=custom_adam, metrics=['accuracy'])
    
    # Catat waktu mulai biar tau mana yang kerjanya paling cepet!
    start_time = time.time()
    
    # 2. Mulai Training
    history = model.fit(
        X_train_dl,
        y_train_dl,
        epochs=20,           
        batch_size=32,
        validation_data=(X_test_dl, y_test_dl),
        class_weight=class_weight, # Pastikan variabel ini namanya sama kayak di kodemu sebelumnya ya
        callbacks=[early_stop],
        verbose=1           # Show loading bar
    )
    
    # Counting clear time
    end_time = time.time()
    lama_waktu = end_time - start_time
    
    # Save history loss & accuracy
    history_dict[name] = history
    
    # Pakai argmin untuk mencari index val_loss terendah (+1 karena urutan epoch mulai dari 1)
    best_epoch = np.argmin(history.history['val_loss']) + 1
    print(f"🌟 Best epoch untuk {name}: {best_epoch} 🌟")
    
    # ==========================================
    # 3. Evaluation MODEL in X_test_dl
    # ==========================================
    print(f"\n📊 Evaluasi Hasil Ujian untuk {name}:")
    
    # Prediksi data test
    y_pred_prob = model.predict(X_test_dl)
    
    # Ubah desimal jadi label tegas
    y_pred_label = (y_pred_prob > 0.5).astype(int)
    
    # Tampilkan Nilai Rapornya
    acc = accuracy_score(y_test_dl, y_pred_label)
    print(f"⏱ Waktu Training : {lama_waktu:.2f} seconds")
    print(f"🏆 Akurasi Global : {acc * 100:.2f}%")
    print("Rapor Detail (Classification Report):")
    print(classification_report(y_test_dl, y_pred_label))
    print("=" * 50)

In [ ]:
# 1. AMBIL JUARA KITA
best_dl_model = dl_models["CNN 1D"]

# 2. BIKIN KALIMAT JEBAKAN (Ujian Dadakan)
tes_tweet_baru = [
    "This product is garbage. Worst customer service ever, I hate it!", 
    
    "You are a disgusting racist nazi bigot, get out of my country!", 
    
    "You are so stupid and ugly, completely useless!",
    
    "White supremacy is the only way to save our nation from those minorities."
]

# 3. PROSES TEKS (Ganti TF-IDF dengan Tokenizer + Padding)
# Ubah teks jadi angka
sekuens_tes = tokenizer.texts_to_sequences(tes_tweet_baru)
# Samakan panjangnya pakai MAX_LEN
X_tes_pad = pad_sequences(sekuens_tes, maxlen=MAX_LEN, padding='post', truncating='post')

# 4. SURUH LSTM MENEBAK!
print("=== HASIL UJIAN DADAKAN AI (LSTM) ===\n")

# Model akan ngeluarin persentase keyakinan (probabilitas 0.0 sampai 1.0)
prediksi_prob = best_dl_model.predict(X_tes_pad)

for teks, prob in zip(tes_tweet_baru, prediksi_prob):
    # Kalau keyakinan AI di atas 50% (0.5), kita anggap Hate Speech
    hasil_akhir = 1 if prob[0] > 0.5 else 0
    label_teks = "🔴 NEGATIF / HATE SPEECH" if hasil_akhir == 1 else "🟢 POSITIF / NORMAL"
    
    print(f"Tweet: '{teks}'")
    print(f"Predict LSTM : {label_teks}")
    print(f"Probability: {prob[0]*100:.2f}%\n")

## Predict Visualization

In [ ]:
report_lstm = history_dict['CNN 1D'].history

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12,5))
ax, ax2 = axes.flatten()

ax.plot(report_lstm['accuracy'], label='Train_Accuracy', color='blue', marker='o')
ax.plot(report_lstm['val_accuracy'], label='Val_Accuracy', color='orange', marker='o')
ax.set_title('Learning Curve: Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, linestyle='--', alpha=.7)

ax2.plot(report_lstm['loss'], label='Train_Loss', color='blue', marker='o')
ax2.plot(report_lstm['val_loss'], label='Val_Loss', color='orange', marker='o')
ax2.set_title('Learning Curve: Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

fig.tight_layout()
plt.show()

# Testing With Another Dataset (Test Dataset)

In [ ]:
seq_test = tokenizer.texts_to_sequences(df_test['tweet'])

X_val_pad = pad_sequences(seq_test, maxlen=MAX_LEN, padding='post', truncating='post')
pred_prob = best_dl_model.predict(X_val_pad)

pred_label = (pred_prob > 0.5).astype(int)

df_submission['label'] = pred_label

result_file = 'hate_speech_segmentation.csv'
df_submission.to_csv(result_file, index=False)

print(df_submission.head(10))